# Unified XPCS analysis: SAXS, GISAXS, WAXS, and GIWAXS


In [ ]:
# Imports and environment setup

import time

_xpcs_notebook_start = time.time()
_xpcs_cell_start = _xpcs_notebook_start

import glob
import itertools
import json
import shutil
import sys
import warnings
from time import localtime, strftime

from pyCHX.chx_packages import *
from pyCHX._xpcs_pipeline_geometry import (
    build_surface_rois, build_waxs_rois, cal_1d_qr_from_roi, combine_detector_masks,
    get_t_qrc_from_roi, intersect_roi_with_mask,
    load_manual_mask, load_manual_roi, load_surface_roi_config,
    plot_roi_index_rates, plot_surface_roi, plot_xpcs_correlation, resolve_geometry,
    surface_q_maps, validate_roi,
)
#from pyCHX.chx_xpcs_xsvs_jupyter_V1 import *
#from pyCHX.chx_generic_functions import get_roi_mask_qval_qwid_by_shift
#from pyCHX.chx_generic_functions import RemoveHot,create_multi_rotated_rectangle_mask

#sys.path.insert(0, "/nsls2/data/chx/shared/CHX_Software/packages/database_processing/") # Need a better way to include these directories
#sys.path.insert(0, "/nsls2/data/chx/shared/CHX_Software/packages/AutoRun_functions/")
from database_processing import *
from AutoRun_functions import *

%matplotlib notebook
plt.rcParams.update({'figure.max_open_warning': 0})
plt.rcParams.update({'image.origin': 'lower'})
plt.rcParams.update({'image.interpolation': 'none'})

%run /nsls2/data/chx/shared/CHX_Software/packages/environment_management/chx_analysis_setup_test_pyCHX.ipynb
_base_path_ = _base_path_pass_
print(colored("--> overwrote '_base_path_' as %s\n\n"%_base_path_,'cyan'))
hostname= !hostname
if not 'saturn4' in hostname[0]:
    enable_olog_attachments = False
    print(colored("WARNING: currently running on resource %s  => attachments to Olog are NOT possible!"%(hostname[0]),'red'))
    print(colored('To enable Olog attachments, this notebook needs to run on saturn4.nsls2.bnl.gov\nFile -> Hub Control Panel -> Stop my Server -> start server and spawn to "CHX staff @Saturn4"','yellow'))
else:
    enable_olog_attachments = True
    print(colored('hostname: %s -> Olog attachments should be possible'%hostname[0],'green'))

kernel = sys.executable
print(colored(f'sys.executable : {kernel}','green'))


_record_xpcs_cell_timing("01 - Imports and environment setup", _xpcs_cell_start)


## Run controls

Choose `saxs`, `gi_saxs`, `waxs`, or `gi_waxs` with `scat_geometry` below.
If the run start metadata contains `scat_geometry`, it takes precedence. Papermill
can still inject a value for older runs.

For a saved mask or ROI, set `manual_mask_file` or `manual_roi_file`. A saved `.npz`
mask contains `mask` and `orientation="analysis"`; a saved `.npz` ROI contains
`roi_mask`, `q_values` (one row per ROI label), and `orientation="analysis"`.
Include `geometry` in either file to prevent reuse with another mode. Legacy
`.npy` masks and `(roi_mask, qval_dict)` `.pkl` ROIs require an explicit
`manual_mask_orientation` or `manual_roi_orientation` set to `"analysis"`.
The image and saved inputs must use the detector orientation after loading.

Beam-mask polygon files are selected by mode as `XPCS_beam_mask_<mode>.json` in
the user masks folder. SAXS also accepts the existing `XPCS_beam_mask.json`.
Set `beam_mask_config_file` for another polygon file or `manual_mask_file` for
a complete saved beam mask. Surface modes require a calibrated incident angle,
beam center, pixel size, sample-detector distance, and wavelength; set the
corresponding `surface_*` controls when metadata lacks them.


In [ ]:
# Run controls

_xpcs_cell_start = time.time()

scat_geometry = 'saxs'  # saxs, gi_saxs, waxs, or gi_waxs; run metadata takes precedence


roi_auto =   True #False if True, will automatically create a roi based on the roi_type ( iso/aniso etc), currently only works for SAXS
fit_peak = False #True

analysis_type_auto = True  #if True, will take "analysis type" option from data acquisition func series
qphi_analysis =   False   #if True, will do q-phi (anisotropic analysis for transmission saxs)

isotropic_Q_mask = 'wide' #'wide' # 'normal' # 'wide'  ## select wich Q-mask to use for rings: 'normal' or 'wide'
phi_Q_mask = 'phi_4x_20deg' #'phi_4x_20deg' #'phi_12x_30deg'

    
auto_bst_mask = True # True: if custom 'XPCS_beam_mask.json' does NOT exist in .../cycle/user/masks, create a default polygon mask for beamstop
force_compress = False   #force to compress data 
use_local_disk = True
bin_frame = False   #generally make bin_frame as False
bin_frame_number = 4 # ignored if bin_frame == False
para_compress = True    #parallel compress
# run_fit_form = False    #run fit form factor -> to be removed
mask_outliers = False  #!!
run_waterfall =  False #True #False #True   #run waterfall analysis
run_profile_plot = False  #run prolfile plot for gi-saxs
run_t_ROI_Inten = True  #run  ROI intensity as a function of time
run_get_mass_center = False  # Analysis for mass center of reflective beam center
run_invariant_analysis = False
run_one_time =  True  #run  one-time
cal_g2_error =  False  #True  #calculate g2 signal to noise
run_fit_g2 = False       #run  fit one-time, the default function is "stretched exponential"
fit_g2_func = 'stretched'
run_two_time =   True    #run  two-time
run_four_time = False #True #True #False   #run  four-time
run_xsvs= False # True #False         #run visibility analysis
att_pdf_report = True    #attach the pdf report to CHX olog
qth_interest = 7 #the intested single qth     

use_sqnorm = True    #if True, use sq to normalize intensity  -> what happens if both are true or false?
use_SG = True  #if True, use the Sawitzky-Golay filter of the avg_img for normalization
use_SG_bin_frames =  False   #if True, use the Sawitzky-Golay filter of the (binned) frame for normalization 

use_imgsum_norm= True  #if True use imgsum to normalize intensity for one-time calculatoin -> else it's doing what?
pdf_version='_%s'%strftime("%Y-%m-%d_%H-%M-%S", localtime())  #for pdf report name
run_dose = False #run dose_depend analysis


# Optional saved inputs use the oriented analysis-image convention. .npz inputs
# carry orientation='analysis'; legacy .npy/.pkl inputs need the explicit
# orientation setting below. ROI .npz files contain roi_mask and q_values.
manual_roi_file = None
surface_roi_config_file = None  # optional JSON recipe for gi_saxs or gi_waxs ROIs
manual_mask_file = None
beam_mask_config_file = None  # optional mode-specific polygon JSON
manual_roi_orientation = None
manual_mask_orientation = None

# For surface modes, set an incident angle here if metadata has no inc_ang,
# incident_angle, or incident_angle_deg. Angles are degrees.
surface_incident_angle_deg = None
surface_beam_center_x = None  # oriented image column, only used if metadata lacks a center
surface_beam_center_y = None  # oriented image row
surface_pixel_size_mm = None
surface_detector_distance_mm = None
surface_wavelength_angstrom = None
surface_inplane_angle_deg = 0.0
gisaxs_q_parallel_edges = None
gisaxs_q_z_edges = None
giwaxs_q_parallel_edges = None
giwaxs_q_z_edges = None

rawdata_link = False # linking raw data in a user sub-folder [sub-folder needs to be created first]

# Geometry is resolved after Papermill parameters and run metadata are loaded.

_record_xpcs_cell_timing("02 - Run controls", _xpcs_cell_start)


### For 'manual' runs: specify uid

In [ ]:
# Manual run setup

_xpcs_cell_start = time.time()

uid = 201401  #changed 29 May 2026 to an existing Eiger series - AF

#####################################################################################################################
md=get_meta_data(uid)
username = md['user']  #this will be dynamically asigned as a parameter
cycle = md['cycle']
try:
    users = md['user_group']
except:
    users = []
    
q_mask_name = '' # do NOT specify, will be created based on Q-Phi mask used and become part of hdf5 filename
track_progress = False # IF run by papermill with progress tracking, will automatically switch to 'True'

_record_xpcs_cell_timing("03 - Manual run setup", _xpcs_cell_start)

# Start the next input-cell timer before Papermill injects parameters.
_xpcs_parameters_start = time.time()
_xpcs_parameters_recorded = False


## Papermill inserts into the cell below:

In [ ]:
# Parameters

# Papermill replaces this cell when parameters are injected. Input cell 05
# records its elapsed time in that case.
_xpcs_cell_start = _xpcs_parameters_start
_record_xpcs_cell_timing("04 - Papermill parameters", _xpcs_cell_start)
_xpcs_parameters_recorded = True


### creating directory for saving output data 

In [ ]:
# Refresh data-session metadata

if not _xpcs_parameters_recorded:
    _record_xpcs_cell_timing("04 - Papermill parameters", _xpcs_parameters_start)
    _xpcs_parameters_recorded = True
_xpcs_cell_start = time.time()

# Data security modification: username for file creation is now the data_session. Should we fix this in the papermill level?
md=get_meta_data(uid)
scat_geometry = resolve_geometry(md, scat_geometry)
md['scat_geometry'] = scat_geometry
if scat_geometry != 'saxs':
    qphi_analysis = False
    roi_auto = False
    use_sqnorm = False
    run_xsvs = False
    run_invariant_analysis = False
if scat_geometry != 'gi_saxs':
    run_profile_plot = False
scat_geometry_ = {'gi_waxs': 'gi_saxs', 'waxs': 'gi_waxs'}.get(scat_geometry, scat_geometry)
print('scattering geometry:', scat_geometry)
username = md['data_session']

_record_xpcs_cell_timing("05 - Refresh data-session metadata", _xpcs_cell_start)


In [ ]:
# Create results directory

_xpcs_cell_start = time.time()

default_dir = _base_path_
data_dir0  = create_user_folder(cycle, username, default_dir if alternate_directory is None else alternate_directory)
md=get_meta_data(uid)
uid=md['uid']
print('The current uid for analysis is: %s...'%uid[:8])
data_dir = os.path.join(data_dir0, '%s/'%uid)
os.makedirs(data_dir, exist_ok=True)
print('Results from this analysis will be stashed in the directory %s' % data_dir)
uidstr = 'uid=%s'%uid

_record_xpcs_cell_timing("06 - Create results directory", _xpcs_cell_start)


### set up progress tracking

In [ ]:
# Set up progress tracking

_xpcs_cell_start = time.time()

if 'process_id' in locals(): #if process_id has been inserted by papermill
    settings_dict_={'run_one_time':run_one_time,'run_two_time':run_two_time,'xsvs':run_xsvs}
    prog_dict_file = get_progress_dict_filename(uid,md['cycle'],md['user'],default_dir)
    track_progress = create_new_prog_dict_run_XPCS_SAXS(prog_dict_file=prog_dict_file,process_id=process_id,settings_dict=settings_dict_, verbose=True)
if track_progress:
    print('going to track progress in %s '%prog_dict_file)
    update_prog_dict_run_XPCS_SAXS(prog_dict_file,process_id,progress_key='Load data and compress', progress='start',start_time=time.time(), stop_time=None, verbose=True)
else: print('Not tracking progress...notebook is likely being run outside of papermill')

_record_xpcs_cell_timing("07 - Set up progress tracking", _xpcs_cell_start)


In [ ]:
# Resolve source data files

_xpcs_cell_start = time.time()

sud = get_sid_filenames_v3(db[uid])
data_fullpath = sud[2]

print ('scan_id, full-uid, data path are:  %s--%s--%s'%(sud[0], sud[1], data_fullpath ))

_record_xpcs_cell_timing("08 - Resolve source data files", _xpcs_cell_start)


# Don't Change the lines below here

* get metadata

In [ ]:
# Load metadata and detector list

_xpcs_cell_start = time.time()

taus=None;g2=None;tausb=None;g2b=None;g12b=None;taus4=None;g4=None;times_xsv=None;contrast_factorL=None; lag_steps = None 

md = get_meta_data( uid )
scat_geometry = resolve_geometry(md, scat_geometry)
md['scat_geometry'] = scat_geometry
md_blue = md.copy()
detectors = sorted(get_detectors(db[uid]))
print('The detectors are:%s'%detectors)
if len(detectors) >1:
    md['detector'] = detectors[1]
    print( md['detector'])

_record_xpcs_cell_timing("09 - Load metadata and detector list", _xpcs_cell_start)


In [ ]:
# Find beam center

_xpcs_cell_start = time.time()

if 'eiger4m' in md['detector'] or md['detector'] == 'image':    
    reverse= True
    rot90= False
elif ('eiger500k' in md['detector']) or ('eiger500K' in md['detector']):    
    reverse= True
    rot90=True
elif 'eiger1m' in md['detector']:    
    reverse= True
    rot90=False
print('Image reverse: %s\nImage rotate 90: %s'%(reverse, rot90))  
try:
    cx , cy = md_blue['beam_center_x'], md_blue['beam_center_y']
    print('(cx,cy) = (%s,%s)'%(cx,cy))
except:
    print('Will find cx,cy later.')

_record_xpcs_cell_timing("10 - Find beam center", _xpcs_cell_start)


### Load ROI defined by "XPCS_Setup" Pipeline

#### Define data analysis type

In [ ]:
# Select analysis type and ROI mask

_xpcs_cell_start = time.time()
if scat_geometry == 'saxs':
    mask_list = check_roi_masknames(_mask_path_+'ROI_masks/')
    iso_masks = [item for item in mask_list if 'phi' not in item]
    phi_masks = [item for item in mask_list if 'phi' in item]
    if analysis_type_auto and 'analysis' in md:
        requested = md['analysis']
        if requested in phi_masks:
            qphi_analysis, phi_Q_mask = True, requested
        elif requested in iso_masks:
            qphi_analysis, isotropic_Q_mask = False, requested
        else:
            qphi_analysis = requested not in ('iso', '', None)
    scat_geometry_ = 'ang_saxs' if qphi_analysis else 'saxs'
else:
    qphi_analysis = False
    scat_geometry_ = {'gi_waxs': 'gi_saxs', 'waxs': 'gi_waxs'}.get(scat_geometry, scat_geometry)
    gisaxs_inc_type = md.get('analysis')
print('analysis geometry:', scat_geometry_, 'qphi:', qphi_analysis)
_record_xpcs_cell_timing("11 - Select analysis type and ROI mask", _xpcs_cell_start)


## Load Data

In [ ]:
# Load detector images

_xpcs_cell_start = time.time()

try: # preferred: load images as dask_array
    imgs,img_md=load_dask_data(uid,md['detector'], _mask_path_+'pixel_masks/',reverse=reverse,rot90=rot90)
    print(colored('SUCCESSFULLY LOADED DATA AS dask-array!','green'))
except:
    print(colored('COULD NOT LOAD DATA AS dask-array, reverting to slow data loading....','red'))
    imgs = load_data(uid,md['detector'],reverse=reverse,rot90=rot90)
    img_md=imgs.md
    print(colored('COULD NOT LOAD DATA AS dask-arry, falling back to slow data loading....','red'))
imgs

_record_xpcs_cell_timing("12 - Load detector images", _xpcs_cell_start)


In [ ]:
# Apply image metadata

_xpcs_cell_start = time.time()

md.update( img_md )
missing_center = 'beam_center_x' not in md or 'beam_center_y' not in md
if missing_center and scat_geometry in ('gi_saxs', 'gi_waxs'):
    if surface_beam_center_x is None or surface_beam_center_y is None:
        raise ValueError('Missing surface beam center in metadata; set surface_beam_center_x and surface_beam_center_y')
    md['beam_center_x'] = surface_beam_center_x
    md['beam_center_y'] = surface_beam_center_y
md['number of images']  = int(md['number of images'])
pixel_mask=np.ones( np.shape(imgs[0]))
if not missing_center and reverse and not rot90:
    md['beam_center_y']=np.shape(imgs[0])[0]- md['beam_center_y']
if not missing_center and reverse and rot90:
    xc=md['beam_center_y'];yc=md['beam_center_x']
    yc=np.shape(imgs[0])[0]-yc
    xc=np.shape(imgs[0])[1]-xc
    md['beam_center_y']=yc;md['beam_center_x']=xc
if rot90 and not reverse:
    raise Exception('ERROR: somebody still needs to figure out the transformation for reverse=%s and rot90=%s!'%(reverse,rot90))
print('Beam Center: xc: %s  yc: %s'%(md['beam_center_x'],md['beam_center_y']))

_record_xpcs_cell_timing("13 - Apply image metadata", _xpcs_cell_start)


### create aggregate image of OAV images

In [ ]:
# Aggregate OAV images

_xpcs_cell_start = time.time()

save_oavs = False
found_oavs = False
for d in detectors:
    if 'OAV_image' in d:
        found_oavs=True   
    
if found_oavs:
    try:
        save_oavs_tifs_v2(  uid, data_dir )
        save_oavs = True
        print('successfully created aggregate image from individual OAV images')
    except:
        print('Failed to create aggregate image from individual OAV images -> OAV images will not be attached to Olog.')

_record_xpcs_cell_timing("14 - Aggregate OAV images", _xpcs_cell_start)


### load detector mask

In [ ]:
# Load detector mask

_xpcs_cell_start = time.time()

detector_mask_path = _mask_path_+'pixel_masks/'
detector_mask_filename = 'detector_mask_%s.json'%md['detector'].split('_')[0]
with open(detector_mask_path+detector_mask_filename) as stream:
    det_mask_dict=json.load(stream)
det_mask=combine_detector_masks(det_mask_dict, imgs[0].shape, md['detector'])

_record_xpcs_cell_timing("15 - Load detector mask", _xpcs_cell_start)


### load beam mask dictionary and re-create polygon mask for current detector and direct beam position

In [ ]:
# Load beam-mask configuration for the selected scattering geometry.
_xpcs_cell_start = time.time()
mask_path = default_dir+'%s/%s/masks/'%(cycle, username)
if beam_mask_config_file is not None:
    beam_candidates = [str(beam_mask_config_file)]
else:
    beam_candidates = [os.path.join(mask_path, 'XPCS_beam_mask_%s.json'%scat_geometry)]
    if scat_geometry == 'saxs':
        beam_candidates.append(os.path.join(mask_path, 'XPCS_beam_mask.json'))
beam_mask_source = next((path for path in beam_candidates if os.path.isfile(path)), None)
if beam_mask_source is not None:
    with open(beam_mask_source) as stream:
        polygon_dict = json.load(stream)
    if not isinstance(polygon_dict, dict) or not isinstance(polygon_dict.get('masks'), dict):
        raise ValueError('Beam-mask configuration must contain a masks dictionary: %s'%beam_mask_source)
    print('loading %s beam mask from %s'%(scat_geometry, beam_mask_source))
elif beam_mask_config_file is not None:
    raise FileNotFoundError('Beam-mask configuration not found: %s'%beam_mask_config_file)
elif auto_bst_mask or manual_mask_file is not None:
    beam_mask_source = 'generated:%s:beam_mask'%scat_geometry
    if scat_geometry == 'saxs':
        polygon_dict={'masks':{'partial_mask':{'x_off': [0, -80], 'y_off': [0, 0],
            'wy_left': [20, 0], 'wy_right': [10, 36], 'wx_up': [20, 0],
            'wx_down': [20, 0], 'center_radius': [0, 0]}}}
    elif scat_geometry == 'gi_saxs':
        polygon_dict={'masks':{'partial_mask':{'x_off': [0, 0], 'y_off': [0, 0],
            'wy_left': [0, 0], 'wy_right': [0, 0], 'wx_up': [36, 0],
            'wx_down': [36, 0], 'center_radius': [0, 0]}}}
    else:
        polygon_dict={'masks': {}}
        if manual_mask_file is None:
            warnings.warn('No %s beam-mask configuration found; the generated beam mask is empty. Set beam_mask_config_file or manual_mask_file to exclude the beam shadow.'%scat_geometry, RuntimeWarning)
else:
    raise FileNotFoundError('No %s beam mask; set beam_mask_config_file or manual_mask_file'%scat_geometry)
_record_xpcs_cell_timing("16 - Load beam-mask configuration for the selected scattering geometry.", _xpcs_cell_start)


In [ ]:
# Define polygon mask

_xpcs_cell_start = time.time()

print('polygon masks: (could edit dictionary entries to modify mask here)')
polygon_dict['masks']

_record_xpcs_cell_timing("17 - Define polygon mask", _xpcs_cell_start)


In [ ]:
# Define polygon mask

_xpcs_cell_start = time.time()

if scat_geometry == 'saxs' and beam_mask_source.startswith('generated:') and 'partial_mask' in polygon_dict['masks']:
    polygon_dict['masks']['partial_mask']['wy_right']= [8, 26]
    polygon_dict['masks']['partial_mask']['wx_up']= [8, 0]
    polygon_dict['masks']['partial_mask']['wx_down']= [8, 0]
    polygon_dict['masks']['partial_mask']['wy_left']= [8, 0]
    polygon_dict['masks']['partial_mask']['x_off'] = [0, -63]

_record_xpcs_cell_timing("18 - Define polygon mask", _xpcs_cell_start)


#### re-create multi-polygon mask:

In [ ]:
# Build polygon mask

_xpcs_cell_start = time.time()

polygon_mask=np.ones_like( det_mask , dtype = bool)
if 'rmask' in list(polygon_dict['masks'].keys()):
    rmask_dict={}
    for aa,a in enumerate(polygon_dict['masks']['rmask']['angles']):
        rmask_dict[a] = create_multi_rotated_rectangle_mask( pixel_mask, 
                                center= [ md['beam_center_x'], md['beam_center_y']   ], length=polygon_dict['masks']['rmask']['length'][aa], width=polygon_dict['masks']['rmask']['width'][aa], angles=[a] )

    for aa,a in enumerate(list(rmask_dict.keys())):
        if aa == 0:
            rmask = rmask_dict[a]
        else:
            rmask*=rmask_dict[a]
    polygon_mask*=rmask
if 'partial_mask' in list(polygon_dict['masks'].keys()):
    partialmask_dict={}
    for ii,i in enumerate(range(len(polygon_dict['masks']['partial_mask']['x_off']))[:]): # [:] use all elements in list; [:2] use the first two elements in list, etc. to create masks
        partialmask_dict[ii] = create_cross_mask(  pixel_mask,center= [ md['beam_center_x']+polygon_dict['masks']['partial_mask']['x_off'][i], md['beam_center_y']+polygon_dict['masks']['partial_mask']['y_off'][i]   ], 
                                wy_left= polygon_dict['masks']['partial_mask']['wy_left'][i], wy_right= polygon_dict['masks']['partial_mask']['wy_right'][i], 
                                wx_up= polygon_dict['masks']['partial_mask']['wx_up'][i], wx_down= polygon_dict['masks']['partial_mask']['wx_down'][i],center_radius= polygon_dict['masks']['partial_mask']['center_radius'][i])
        
    for i in list(partialmask_dict.keys()):
        if i == 0:
            partial_mask = partialmask_dict[i]
        else:
            partial_mask*=partialmask_dict[i]
    polygon_mask*=partial_mask
if 'window_mask' in list(polygon_dict['masks'].keys()):
    try: # center_circle has been renamed to center_disk at some point -> with try this works on old and new environements
        window_shadow  = ~create_cross_mask(  pixel_mask, center=[md['beam_center_x'],md['beam_center_y']],
                           wy_left= 0, wy_right= 0, 
                             wx_up= 0, wx_down= 0,center_disk=True, center_radius= polygon_dict['masks']['window_mask']['center_radius'])
    except:
        window_shadow  = ~create_cross_mask(  pixel_mask, center=[md['beam_center_x'],md['beam_center_y']],
                           wy_left= 0, wy_right= 0, 
                             wx_up= 0, wx_down= 0,center_circle=True, center_radius= polygon_dict['masks']['window_mask']['center_radius'])
        print('using "create_cross_mask" for older python environments')
    polygon_mask*=window_shadow

_record_xpcs_cell_timing("19 - Build polygon mask", _xpcs_cell_start)


In [ ]:
# Plot detector and beam masks
_xpcs_cell_start = time.time()
if manual_mask_file is not None:
    polygon_mask = load_manual_mask(manual_mask_file, det_mask.shape, manual_mask_orientation, geometry=scat_geometry)
    mask_input_source = str(manual_mask_file)
else:
    mask_input_source = beam_mask_source
mask = polygon_mask * det_mask
mask_copy = mask.copy()
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('detector mask: %s\nbeam mask: %s'%(detector_mask_path+detector_mask_filename, mask_input_source))
for axis, image, title in zip(ax, (det_mask, polygon_mask, mask),
                              ('detector mask', 'beam mask', 'combined mask')):
    axis.imshow(image)
    axis.set_title(title)
_record_xpcs_cell_timing("20 - Plot detector and beam masks", _xpcs_cell_start)


In [ ]:
# Display metadata

_xpcs_cell_start = time.time()

print_dict( md,  ['suid', 'number of images', 'uid', 'scan_id', 'start_time', 'stop_time', 'sample', 'Measurement',
                  'acquire period', 'exposure time',  
         'det_distance', 'beam_center_x', 'beam_center_y', ] )

_record_xpcs_cell_timing("21 - Display metadata", _xpcs_cell_start)


## Overwrite Some Metadata if Wrong Input

### Define incident beam center (also define reflection beam center for gisaxs)

In [ ]:
# Incident beam center in the oriented analysis image (column, row).

_xpcs_cell_start = time.time()
inc_x0 = float(md['beam_center_x'])
inc_y0 = float(md['beam_center_y'])
print('incident beam center:', inc_x0, inc_y0)

_record_xpcs_cell_timing("22 - Incident beam center in the oriented analysis image (column, row).", _xpcs_cell_start)


In [ ]:
# Validate experiment metadata

_xpcs_cell_start = time.time()

if scat_geometry in ('gi_saxs', 'gi_waxs'):
    if 'x_pixel_size' not in md:
        if surface_pixel_size_mm is None:
            raise ValueError('Missing surface pixel size in metadata; set surface_pixel_size_mm')
        md['x_pixel_size'] = float(surface_pixel_size_mm) / 1000
    if 'det_distance' not in md and 'detector_distance' not in md:
        if surface_detector_distance_mm is None:
            raise ValueError('Missing surface detector distance in metadata; set surface_detector_distance_mm')
        md['det_distance'] = float(surface_detector_distance_mm) / 1000
    if 'wavelength' not in md and 'incident_wavelength' not in md:
        if surface_wavelength_angstrom is None:
            raise ValueError('Missing surface wavelength in metadata; set surface_wavelength_angstrom')
        md['wavelength'] = float(surface_wavelength_angstrom)
dpix, lambda_, Ldet,  exposuretime, timeperframe, center = check_lost_metadata(
    md, md['number of images'], inc_x0 = inc_x0, inc_y0=   inc_y0, pixelsize = 7.5*10*(-5) )
if scat_geometry in ('gi_saxs', 'gi_waxs'):
    surface_maps = surface_q_maps(
        mask.shape, (inc_x0, inc_y0), dpix, Ldet, lambda_, md,
        incident_angle_deg=surface_incident_angle_deg,
        inplane_angle_deg=surface_inplane_angle_deg,
    )
    qr_map, qz_map = surface_maps['q_parallel'], surface_maps['q_z']
    refl_x0, refl_y0 = surface_maps['reflected_center']
    md['incident_angle_deg'] = surface_maps['incident_angle_deg']
    md['inplane_angle_deg'] = surface_maps['inplane_angle_deg']

setup_pargs=dict(uid=uidstr, dpix= dpix, Ldet=Ldet, lambda_= lambda_, exposuretime=exposuretime,
        timeperframe=timeperframe, center=center, path= data_dir)
print_dict( setup_pargs )

_record_xpcs_cell_timing("23 - Validate experiment metadata", _xpcs_cell_start)


In [ ]:
# Apply detector mask

_xpcs_cell_start = time.time()

show_img(np.array(mask,dtype=int),image_name = uidstr + '_mask', save=False, path=data_dir, aspect=1, center=center)
mask_load=mask.copy()

imgsa = apply_mask( imgs, np.array(mask,dtype=int) )

_record_xpcs_cell_timing("24 - Apply detector mask", _xpcs_cell_start)


## Check several frames average  intensity

#### Load ROI mask depending on data analysis type

In [ ]:
# Load and align ROI mask

_xpcs_cell_start = time.time()
roi_date = 'N.A'
roi_input_source = None
if scat_geometry == 'saxs':
    q_mask_name = phi_Q_mask if qphi_analysis else isotropic_Q_mask
    if manual_roi_file is not None or not roi_auto:
        if manual_roi_file is None:
            raise ValueError('Set manual_roi_file when roi_auto is False')
        fp = str(manual_roi_file)
        roi_mask, qval_dict = load_manual_roi(
            fp, mask.shape, scat_geometry, orientation=manual_roi_orientation,
            mask=mask, center=(inc_x0, inc_y0), dpix=dpix,
            distance=Ldet, wavelength=lambda_)
        roi_input_source = fp
    else:
        fp = _mask_path_+'ROI_masks/general_roi_mask_%s.json'%q_mask_name
        with open(fp) as stream:
            mask_dict = json.loads(json.load(stream))
        roi_mask0 = np.array(mask_dict['roi_mask'])
        old_cen = mask_dict['original center']
        if 'eiger500k' in md['detector'] or 'eiger4m' in md['detector']:
            setup_pargs['center'] = center[::-1]
        roi_mask, qval_dict, qwid_dict = get_roi_mask_qval_qwid_by_shift(
            new_cen=[0, 0], new_mask=mask, old_cen=old_cen,
            old_roi_mask=roi_mask0, limit_qnum=180,
            setup_pargs=setup_pargs, geometry=scat_geometry_)
        setup_pargs['center'] = center
        roi_input_source = fp
    roi_mask2 = roi_mask.copy()
elif scat_geometry in ('gi_saxs', 'gi_waxs'):
    q_mask_name = scat_geometry
    parallel_edges = gisaxs_q_parallel_edges if scat_geometry == 'gi_saxs' else giwaxs_q_parallel_edges
    z_edges = gisaxs_q_z_edges if scat_geometry == 'gi_saxs' else giwaxs_q_z_edges
    use_roi_profiles = (surface_roi_config_file is not None or manual_roi_file is not None
                        or parallel_edges is not None or z_edges is not None)
    surface_rois = None
    if surface_roi_config_file is not None:
        if manual_roi_file is not None:
            raise ValueError('Set either surface_roi_config_file or manual_roi_file, not both')
        fp = str(surface_roi_config_file)
        roi_mask, qval_dict = load_surface_roi_config(
            fp, mask.shape, scat_geometry, mask, surface_maps)
        roi_input_source = fp
    elif manual_roi_file is not None:
        fp = str(manual_roi_file)
        roi_mask, qval_dict = load_manual_roi(
            fp, mask.shape, scat_geometry, orientation=manual_roi_orientation,
            mask=mask, q_maps=surface_maps)
        roi_input_source = fp
    else:
        surface_rois = build_surface_rois(qr_map, qz_map, mask,
            parallel_edges=parallel_edges, z_edges=z_edges,
            parallel_count=8 if scat_geometry == 'gi_saxs' else 12,
            z_count=3 if scat_geometry == 'gi_saxs' else 4)
        roi_mask, qval_dict = surface_rois['roi_mask'], surface_rois['qval_dict']
        fp = 'generated:%s:%s'%(scat_geometry, md['uid'])
        roi_input_source = fp
    Qr, Qz = (None, None) if use_roi_profiles else (surface_rois['Qr'], surface_rois['Qz'])
    Qrs, Qzs = Qr, Qz
    roi_masks, qval_dicts = roi_mask.copy(), qval_dict.copy()
    ticks = None
    validate_roi(roi_mask, qval_dict, mask.shape, scat_geometry,
                 mask=mask, q_maps=surface_maps)
    roi_mask2 = roi_mask.copy()
else:
    # WAXS intensity bands use the compressed average image, available in cell 52.
    q_mask_name = 'waxs'
    roi_mask = None
    roi_mask2 = None
    fp = 'generated:waxs:%s'%md['uid']
_record_xpcs_cell_timing("25 - Load and align ROI mask", _xpcs_cell_start)


In [ ]:
# Display ROI mask

_xpcs_cell_start = time.time()

#roi_mask = shift_mask(roi_mask, 10,30)  #if shift mask to get new mask
if roi_mask is not None:
    show_img(roi_mask, aspect=1.0, image_name = fp, center=center,vmin=0)

_record_xpcs_cell_timing("26 - Display ROI mask", _xpcs_cell_start)


In [ ]:
# Correlate sample average image

_xpcs_cell_start = time.time()

img_choice_N = 5
img_samp_index = random.sample(  range(md['number of images']), img_choice_N) 
avg_img =  get_avg_img( imgsa, img_samp_index, plot_ = False, uid =uidstr)
if avg_img.max() == 0:
    print('There are no photons recorded for this uid: %s'%uid)
    print('The data analysis should be terminated! Please try another uid.')

_record_xpcs_cell_timing("27 - Correlate sample average image", _xpcs_cell_start)


In [ ]:
# Display sample image

_xpcs_cell_start = time.time()

show_img( imgsa[ 5],  vmin = .01, vmax = 10, logs=False, aspect=1, #save_format='tif',
         image_name= uidstr + '_img_avg',  save=True, path=data_dir, cmap=cmap_albula,center=center)

_record_xpcs_cell_timing("28 - Display sample image", _xpcs_cell_start)


## Compress Data
* Generate a compressed data with filename
* Replace old mask with a new mask with removed hot pixels
* Do average image
* Do each image sum
* Find badframe_list for where image sum above bad_pixel_threshold
* Check shutter open frame to get good time series


In [ ]:
# Calculate photon occupancy

_xpcs_cell_start = time.time()

compress=True
photon_occ = len( np.where(avg_img)[0] ) / ( np.shape(imgsa[0])[0]*np.shape(imgsa[0])[1]*md['number of images'])
#compress =  photon_occ < .4  #if the photon ocupation < 0.5, do compress
print ("The non-zeros photon occupation is %s."%( photon_occ))
print("Will " + 'Always ' + ['NOT', 'DO'][compress]  + " apply compress process.")

_record_xpcs_cell_timing("29 - Calculate photon occupancy", _xpcs_cell_start)


In [ ]:
# Define good start

_xpcs_cell_start = time.time()

if  "eiger4m" in md['detector'] or md['detector'] == 'image':    
    good_start =    0 #make the good_start at least 0
elif ("eiger500k" in md['detector']) or ("eiger500K" in md['detector']): 
    good_start = 0#100  #5  #make the good_start at least 0
elif "eiger1m" in md['detector'] or md['detector'] == 'image':    
    good_start =    0    
    
if bin_frame:
    bin_frame_number=bin_frame_number
    acquisition_period = md['acquire period']
    timeperframe = acquisition_period * bin_frame_number
else:
    bin_frame_number =1
print('first frame to be used: %s\nframe binning: %s'%(good_start,bin_frame_number))

_record_xpcs_cell_timing("30 - Define good start", _xpcs_cell_start)


### change compression behavior if necessary

In [ ]:
# Set compression behavior

_xpcs_cell_start = time.time()

#force_compress = False
#use_local_disk = True
print('force compression: %s  use local disk: %s'%(force_compress,use_local_disk))

_record_xpcs_cell_timing("31 - Set compression behavior", _xpcs_cell_start)


In [ ]:
# Compress detector data

_xpcs_cell_start = time.time()

imgs_=np.ones(md['number of images']) #just need to fake an array of appropriate length for the compression routine


# if not use_local_disk:
#     cmp_path = '/nsls2/data/chx/legacy/Compressed_Data'
# else:
#     cmp_path = '/tmp_data/compressed'
#     print(here)
    
    
cmp_path = data_dir0[:-8] + 'Compressed_Data' 
if bin_frame_number==1:   
    cmp_file = '/uid_%s.cmp'%md['uid']
else:
    cmp_file = '/uid_%s_bined--%s.cmp'%(md['uid'],bin_frame_number)

#force_compress = True
filename = cmp_path + cmp_file  
mask2, avg_img, imgsum, bad_frame_list = compress_eigerdata(imgs_, det_mask, md, filename,   ########## NEW: mask -> det_mask
         force_compress= force_compress,  para_compress= para_compress,  bad_pixel_threshold = 1e14,
                                    reverse=reverse, rot90=rot90, nobytes=4,
                        bins=bin_frame_number, num_sub= 100, num_max_para_process= 500, with_pickle=True,
                        direct_load_data =use_local_disk, data_path = data_fullpath, new_path = '/tmp/' )                                  
                                                         
if scat_geometry == 'waxs':
    if manual_roi_file is not None:
        fp = str(manual_roi_file)
        roi_mask, qval_dict = load_manual_roi(
            fp, mask.shape, scat_geometry, orientation=manual_roi_orientation, mask=mask)
        roi_input_source = fp
    else:
        roi_mask, qval_dict = build_waxs_rois(avg_img, mask)
        roi_input_source = fp
    roi_mask2 = roi_mask.copy()
    validate_roi(roi_mask, qval_dict, mask.shape, scat_geometry, mask=mask)
    show_img(roi_mask, aspect=1.0, image_name=uidstr+'_waxs_ROI', center=center, vmin=0)

min_inten = 10    
good_start = max(good_start, np.where( np.array(imgsum) > min_inten )[0][0] )    
print ('The good_start frame number is: %s '%good_start)

FD = Multifile(filename, good_start, len(imgs_)//bin_frame_number )


uid_ = uidstr + '_fra_%s_%s'%(FD.beg, FD.end)
print( uid_ )
plot1D( y = imgsum[ np.array( [i for i in np.arange(good_start, len(imgsum)) if i not in bad_frame_list])],
       title =uidstr + '_imgsum', xlabel='Frame', ylabel='Total_Intensity', legend='imgsum'   )
Nimg = int( md['number of images']/bin_frame_number )

# mask2 =  mask * pixel_mask * Chip_Mask
mask_copy = mask.copy()
mask_copy2 = mask.copy()

#avg_img *= mask

show_img( avg_img,  vmin=1e-1, vmax= 1e2, logs=True, aspect=1, #save_format='tif',
         image_name= uidstr + '_img_avg',  save=True, 
         path=data_dir, center=center, cmap = cmap_albula )

_record_xpcs_cell_timing("32 - Compress detector data", _xpcs_cell_start)


### Linking Raw data in 'rawdata' folder [folder needs to be created first]

In [ ]:
# Link raw data

_xpcs_cell_start = time.time()

if(rawdata_link):
    #data_fullpath - h5 file
    #filename - cmp file
    rawdata_folder = data_dir0[:-8]+'rawdata'
    h5_linkname = rawdata_folder+'/'+cmp_file[:-4]+'.h5'
    
    #print('The raw data directory is: %s'%rawdata_folder)
    #print('The hdf5 file is: %s'%data_fullpath)
    #print('The cmp file is: %s'%filename[:])
    #print('The h5 link name is: %s'%h5_linkname)
    !ln -s {data_fullpath} {h5_linkname}
    !ln -s {filename} {rawdata_folder+'/'}

_record_xpcs_cell_timing("33 - Link raw data", _xpcs_cell_start)


In [ ]:
# Update load/compression progress

_xpcs_cell_start = time.time()

if track_progress:
    update_prog_dict_run_XPCS_SAXS(prog_dict_file,process_id,progress_key='Load data and compress', progress='stop',start_time=None, stop_time=time.time(), verbose=True)
    if settings_dict_['run_one_time']:
        update_prog_dict_run_XPCS_SAXS(prog_dict_file,process_id,progress_key='run_one_time', progress='start',start_time=time.time(), stop_time=None, verbose=True)

_record_xpcs_cell_timing("34 - Update load/compression progress", _xpcs_cell_start)


## Get bad frame list by a polynominal fit

In [ ]:
# Set compressed-data range

_xpcs_cell_start = time.time()

good_end= None # 2000  
if good_end is not None:
    FD = Multifile(filename, good_start, min( len(imgs_)//bin_frame_number, good_end) )
    uid_ = uidstr + '_fra_%s_%s'%(FD.beg, FD.end)
    print( uid_ )

_record_xpcs_cell_timing("35 - Set compressed-data range", _xpcs_cell_start)


In [ ]:
# Optional refine start frame

_xpcs_cell_start = time.time()

re_define_good_start =False
if re_define_good_start:
    good_start = 180
    #good_end = 19700
    good_end = len(imgs_)
    FD = Multifile(filename, good_start, good_end) 
    uid_ = uidstr + '_fra_%s_%s'%(FD.beg, FD.end)
    print( FD.beg, FD.end)

_record_xpcs_cell_timing("36 - Optional refine start frame", _xpcs_cell_start)


In [ ]:
# Identify bad frames

_xpcs_cell_start = time.time()

bad_frame_list =  get_bad_frame_list( imgsum, fit='both',  plot=True,polyfit_order = 30,                                      
                        scale= 3.5,  good_start = good_start, good_end=good_end, uid= uidstr, path=data_dir)

print( 'The bad frame list length is: %s'%len(bad_frame_list) )

_record_xpcs_cell_timing("37 - Identify bad frames", _xpcs_cell_start)


### Create new mask by masking the bad pixels and get new avg_img

In [ ]:
# Save frame intensities

_xpcs_cell_start = time.time()

imgsum_y = imgsum[ np.array( [i for i in np.arange( len(imgsum)) if i not in bad_frame_list])]
imgsum_x = np.arange( len( imgsum_y))
save_lists(  [imgsum_x, imgsum_y], label=['Frame', 'Total_Intensity'],
           filename=uidstr + '_img_sum_t', path= data_dir  )
print('number of bad pixels masked: %s'%len(imgsum_y))

_record_xpcs_cell_timing("38 - Save frame intensities", _xpcs_cell_start)


### Plot time~ total intensity of each frame

In [ ]:
# Plot frame intensities

_xpcs_cell_start = time.time()

plot1D( y = imgsum_y, title = uidstr + '_img_sum_t', xlabel='Frame', c='b',
       ylabel='Total_Intensity', legend='imgsum', save=True, path=data_dir)

_record_xpcs_cell_timing("39 - Plot frame intensities", _xpcs_cell_start)


## Get Dynamic Mask (currently designed for 500K)

# NEED ALL Detectors! There are 3 bit-depth levels!
### check Dectris documentation: 16/8/4 bit(?)

In [ ]:
# Build dynamic detector mask

_xpcs_cell_start = time.time()

if  "eiger4m" in md['detector'] or md['detector'] == 'image':    
    pass
elif ("eiger500k" in md['detector']) or ("eiger500K" in md['detector']):  
    if md['eiger500K_single_cam_acquire_period'] <= 0.00015:  #will check this logic
        if imgs[0].dtype == 'uint8':
            print('Create dynamic mask for 500K due to 9K data acquistion!!!')
            #bdp = find_bad_pixels_FD( bad_frame_list, FD, img_shape = avg_img.shape, threshold= 15 )  
            bdp = find_bad_pixels_FD( np.arange(Nimg), FD, img_shape = avg_img.shape, threshold= 15 )  
            mask = mask_copy2.copy()
            mask *=bdp 
            mask_copy = mask.copy()
            show_img(  np.array(mask,dtype=int), image_name='New Mask_uid=%s'%uid, aspect=1 )

dynamic_mask_applied = not np.array_equal(mask, mask_copy2)
if dynamic_mask_applied and scat_geometry != 'saxs':
    roi_mask, qval_dict = intersect_roi_with_mask(roi_mask, qval_dict, mask, geometry=scat_geometry)
    roi_mask2 = roi_mask.copy()
    print('Dynamic mask updated %s ROIs'%len(qval_dict))

_record_xpcs_cell_timing("40 - Build dynamic detector mask", _xpcs_cell_start)


## SAXS Scattering Geometry

In [ ]:
# Setup center args

_xpcs_cell_start = time.time()

setup_pargs['center'] = center[::-1]

_record_xpcs_cell_timing("41 - Setup center args", _xpcs_cell_start)


In [ ]:
# Calculate SAXS circular average

_xpcs_cell_start = time.time()

if scat_geometry =='saxs':
    outlier_threshold = 5.5 # change here, if necessary
    maximum_outlier_fraction = .1
    
    roi_mask = roi_mask2.copy()
    ## Get circular average| * Do plot and save q~iq
    mask = mask_copy.copy()
    hmask = create_hot_pixel_mask( avg_img, threshold = 2e8, center=center, center_radius= 100)
    mask =np.array( mask * hmask, dtype=bool)
    
    if mask_outliers:
        hhmask = outlier_mask(avg_img,mask,roi_mask,outlier_threshold = outlier_threshold, maximum_outlier_fraction = maximum_outlier_fraction,verbose=False,plot=False)
        mask =np.array( mask * hhmask, dtype=bool)
        
    qp_saxs, iq_saxs, q_saxs = get_circular_average( avg_img * det_mask , mask * hmask, save=True,
    #qp_saxs, iq_saxs, q_saxs = get_circular_average( avg_img * det_mask*mask , mask, save=True,
                                                    pargs=setup_pargs  )
    plot_circular_average( qp_saxs, iq_saxs, q_saxs,  pargs=setup_pargs,  show_pixel=True,
                      xlim=[qp_saxs.min(), qp_saxs.max()*1.0], ylim = [iq_saxs.min(), iq_saxs.max()*2] )
    
    print('Removed %s hot pixels.'%len(np.nonzero(hmask-1)[0]))
    roi_mask, qval_dict = intersect_roi_with_mask(roi_mask, qval_dict, mask, geometry='saxs')

_record_xpcs_cell_timing("42 - Calculate SAXS circular average", _xpcs_cell_start)


In [ ]:
# Validate and plot SAXS ROIs

_xpcs_cell_start = time.time()

if scat_geometry =='saxs':  
    qr = np.array( [qval_dict[k][0] for k in sorted( qval_dict.keys())] )
    if qphi_analysis ==  False:
        try:
            qr_cal, qr_wid = get_QrQw_From_RoiMask( roi_mask, setup_pargs  ) 
            print(len(qr))         
            if (qr_cal - qr).sum() >=1e-3:
                print( 'The loaded ROI mask might not be applicable to this UID: %s.'%uid)
                print('Please check the loaded roi mask file.')
        except:
            print('Something is wrong with the roi-mask. Please check the loaded roi mask file.')            
            
    
    show_ROI_on_image( avg_img*roi_mask, roi_mask, center, label_on = False, rwidth = 600, alpha=.9,  
                 save=True, path=data_dir, uid=uidstr, vmin= 1e-1,
                 vmax= 1e3, #np.max(avg_img),
                 aspect=1,
                 show_roi_edge=True,     
                 show_ang_cor = True) 
    plot_qIq_with_ROI( q_saxs, iq_saxs, np.unique(qr), logs=True, uid=uidstr, 
                      xlim=[q_saxs.min(), q_saxs.max()*1.02],#[0.0001,0.08],
                  ylim = [iq_saxs.min(), iq_saxs.max()*1.02],  save=True, path=data_dir)

_record_xpcs_cell_timing("43 - Validate and plot SAXS ROIs", _xpcs_cell_start)


# Time Depedent I(q) Analysis

In [ ]:
# Calculate time-dependent I(q)

_xpcs_cell_start = time.time()

if scat_geometry =='saxs':
    slice_num=3
    slice_width=2
    Nimg = FD.end - FD.beg 
    time_edge = create_time_slice( Nimg, slice_num=slice_num, slice_width=slice_width, edges = None )
    time_edge =  np.array( time_edge ) + good_start   
    if slice_num*slice_width >10: # doing circular average from sparsified data
        qpt, iqst, qt = get_t_iqc( FD, time_edge, mask*det_mask, pargs=setup_pargs, nx=1500, show_progress= False )
    else: # if we only need a few images, circular average from dask.array is faster
        qpt, iqst, qt = get_t_iqc_imstack( imgs, time_edge, mask*det_mask, pargs=setup_pargs, nx=1500, show_progress= False )
    plot_t_iqc( qt, iqst, time_edge, pargs=setup_pargs, xlim=[qt.min(), qt.max()],
           ylim = [iqst.min(), iqst.max()], save=True )

_record_xpcs_cell_timing("44 - Calculate time-dependent I(q)", _xpcs_cell_start)


## For fit one peak

In [ ]:
# Fit SAXS peak

_xpcs_cell_start = time.time()

if scat_geometry =='saxs':
    if fit_peak:   
        q1 =  0.162
        x = q_saxs
        y= iq_saxs
        aind = [ find_index(  q_saxs, q1 ) ]
        w = 50
        xrange = [  aind[0] -w, aind[0] + w]        
        cen, cen_std, wid, wid_std, xf, out =  fit_one_peak_curve( x, y, xrange ) 
        fp = data_dir + uidstr + '_qIq_fit.png'         
        ax = plot_xy_with_fit(  x, y, xf, out, cen, cen_std,wid, wid_std,
                              xlim=[0.01,0.3],xlabel= 'q ('r'$\AA^{-1}$)',
                    ylabel='I(q)', filename=fp)

_record_xpcs_cell_timing("45 - Fit SAXS peak", _xpcs_cell_start)


In [ ]:
# Calculate scattering invariant

_xpcs_cell_start = time.time()

if run_invariant_analysis:
    if scat_geometry =='saxs':
        invariant = get_iq_invariant( qt, iqst )
        time_stamp = time_edge[:,0] * timeperframe

    if scat_geometry =='saxs':
        plot_q2_iq( qt, iqst, time_stamp,pargs=setup_pargs,ylim=[ -0.001, 0.01] , 
                   xlim=[0.007,0.2],legend_size= 6  )

    if scat_geometry =='saxs':
        plot_time_iq_invariant( time_stamp, invariant, pargs=setup_pargs,  )

    if False:
        iq_int = np.zeros( len(iqst) )
        fig, ax = plt.subplots()
        q = qt
        for i in range(iqst.shape[0]):
            yi = iqst[i] * q**2
            iq_int[i] = yi.sum()
            time_labeli = 'time_%s s'%( round(  time_edge[i][0] * timeperframe, 3) )
            plot1D( x = q, y = yi, legend= time_labeli, xlabel='Q (A-1)', ylabel='I(q)*Q^2', title='I(q)*Q^2 ~ time',
                   m=markers[i], c = colors[i], ax=ax, ylim=[ -0.001, 0.01] , xlim=[0.007,0.2],
                  legend_size=4)

        #print( iq_int )

_record_xpcs_cell_timing("46 - Calculate scattering invariant", _xpcs_cell_start)


# GiSAXS Scattering Geometry

In [ ]:
# Plot calibrated surface q map
_xpcs_cell_start = time.time()
if scat_geometry in ('gi_saxs', 'gi_waxs'):
    plot_surface_roi(avg_img, roi_mask, surface_maps, data_dir, uidstr)
_record_xpcs_cell_timing("47 - Plot calibrated surface q map", _xpcs_cell_start)


## Static Analysis for gisaxs

In [ ]:
# Static GISAXS ROIs are generated from this run's calibrated map.

_xpcs_cell_start = time.time()
if scat_geometry == 'gi_saxs':
    print('GISAXS static ROI count:', len(qval_dicts))

_record_xpcs_cell_timing("48 - Static GISAXS ROIs are generated from this run's calibrated map.", _xpcs_cell_start)


In [ ]:
# Calculate time-dependent GISAXS profiles

_xpcs_cell_start = time.time()

if  scat_geometry =='gi_saxs':    
    Nimg = FD.end - FD.beg 
    time_edge = create_time_slice( N= Nimg, slice_num= 3, slice_width= 2, edges = None )
    time_edge =  np.array( time_edge ) + good_start
    print( time_edge )    
    if use_roi_profiles:
        qrt_pds = get_t_qrc_from_roi(
            FD, time_edge, roi_mask, qval_dict, get_avg_imgc, path=data_dir, uid=uidstr)
    else:
        qrt_pds = get_t_qrc( FD, time_edge, Qrs, Qzs, qr_map, qz_map, mask=mask, path=data_dir, uid = uidstr )
    plot_qrt_pds( qrt_pds, time_edge, qz_index = 0, uid = uidstr, path =  data_dir )

_record_xpcs_cell_timing("49 - Calculate time-dependent GISAXS profiles", _xpcs_cell_start)


# Make a Profile Plot

In [ ]:
# Calculate GISAXS waterfall from a ROI generated for this run.
_xpcs_cell_start = time.time()
if scat_geometry == 'gi_saxs' and run_profile_plot:
    waterfall_roi = np.asarray(roi_mask == 1, dtype=np.int32)
    wat = cal_waterfallc(FD, waterfall_roi, qindex=1,
                         save=True, path=data_dir, uid=uidstr)
_record_xpcs_cell_timing("50 - Calculate GISAXS waterfall from a ROI generated for this run.", _xpcs_cell_start)


In [ ]:
# Plot GISAXS waterfall

_xpcs_cell_start = time.time()

if  scat_geometry =='gi_saxs':
    if run_profile_plot:
        plot_waterfallc( wat, qindex=1, aspect=None, vmin=1, vmax= np.max( wat), uid=uidstr, save =True, 
                        path=data_dir, beg= FD.beg)

_record_xpcs_cell_timing("51 - Plot GISAXS waterfall", _xpcs_cell_start)


## Dynamic Analysis for gi_saxs

In [ ]:
# Calculate GISAXS 1D profile

_xpcs_cell_start = time.time()

if scat_geometry =='gi_saxs':       
    # ROI overlay saved by the private surface plotting adapter.        
    ## Get 1D Curve (Q||-intensity¶)
    if use_roi_profiles:
        qr_1d_pds = cal_1d_qr_from_roi(
            avg_img, roi_mask, qval_dict, path=data_dir, uid=uidstr)
    else:
        qr_1d_pds = cal_1d_qr( avg_img, Qr, Qz, qr_map, qz_map, inc_x0= None, mask=mask, setup_pargs=setup_pargs )
    plot_qr_1d_with_ROI( qr_1d_pds, qr_center=np.unique( np.array(list( qval_dict.values() ) )[:,0] ),
                    loglog=True, save=True, uid=uidstr, path = data_dir)

_record_xpcs_cell_timing("52 - Calculate GISAXS 1D profile", _xpcs_cell_start)


# GiWAXS Scattering Geometry

In [ ]:
# Plot WAXS and GIWAXS ROIs
_xpcs_cell_start = time.time()
if scat_geometry == 'gi_waxs':
    plot_surface_roi(avg_img, roi_mask, surface_maps, data_dir, uidstr)
elif scat_geometry == 'waxs':
    show_ROI_on_image(avg_img, roi_mask, label_on=True, alpha=.5,
                      save=True, path=data_dir, uid=uidstr, vmin=0.1, vmax=500)
_record_xpcs_cell_timing("53 - Plot WAXS and GIWAXS ROIs", _xpcs_cell_start)


* Extract the labeled array

In [ ]:
# Extract ROI pixel labels

_xpcs_cell_start = time.time()

qind, pixelist = roi.extract_label_indices(roi_mask)
noqs = len(np.unique(qind))
print('calculating correlation functions for %s Qs'%noqs)

_record_xpcs_cell_timing("54 - Extract ROI pixel labels", _xpcs_cell_start)


* Number of pixels in each q box

In [ ]:
# Plot ROI pixel counts

_xpcs_cell_start = time.time()

nopr = np.bincount(qind, minlength=(noqs+1))[1:]
roi_x = range(len(nopr))
plt.semilogy(roi_x,nopr,'ro',fillstyle='none');plt.xlabel('ROI #');plt.ylabel('number of pixels in ROI')

_record_xpcs_cell_timing("55 - Plot ROI pixel counts", _xpcs_cell_start)


## Check one ROI intensity

In [ ]:
# Check ROI intensity

_xpcs_cell_start = time.time()

roi_inten = check_ROI_intensity( avg_img, roi_mask, ring_number=min(5, len(qval_dict)), uid =uidstr ) #roi starting from 1

_record_xpcs_cell_timing("56 - Check ROI intensity", _xpcs_cell_start)


## Do a waterfall analysis

In [ ]:
# Calculate waterfall analysis

_xpcs_cell_start = time.time()

qth_interest = min(2, len(qval_dict)) # ROI index starts from 1
if scat_geometry =='saxs' or scat_geometry =='gi_waxs' or scat_geometry == 'gi_saxs' or scat_geometry == 'waxs':
    if run_waterfall:    
        wat = cal_waterfallc( FD, roi_mask, qindex= qth_interest, save =True, path=data_dir, uid=uidstr)
        plot_waterfallc( wat, qth_interest, aspect= None, vmin=1e-1, vmax= wat.max(), uid=uidstr, save =True, 
                        path=data_dir, beg= FD.beg, cmap = cmap_vge )

_record_xpcs_cell_timing("57 - Calculate waterfall analysis", _xpcs_cell_start)


In [ ]:
# Calculate ROI intensities over time

_xpcs_cell_start = time.time()

ring_avg = None    
if run_t_ROI_Inten:
    times_roi, mean_int_sets = cal_each_ring_mean_intensityc(FD, roi_mask, timeperframe = None, multi_cor= False )
    mean_int_setsF = np.zeros( [FD.end, mean_int_sets.shape[1] ] )
    mean_int_setsF[FD.beg:FD.end] =mean_int_sets
    plot_each_ring_mean_intensityc( times_roi, mean_int_sets,  uid = uidstr, save=True, path=data_dir )
    roi_avg = np.average( mean_int_sets, axis=0)

_record_xpcs_cell_timing("58 - Calculate ROI intensities over time", _xpcs_cell_start)


## Analysis for mass center of reflective beam center

In [ ]:
# Calculate ROI mass center

_xpcs_cell_start = time.time()

if run_get_mass_center:
    cx, cy = get_mass_center_one_roi(FD, roi_mask, roi_ind=min(25, len(qval_dict)))
    fig,ax=plt.subplots(2)
    plot1D( cx, m='o', c='b',ax=ax[0], legend='mass center-refl_X', 
           ylim=[940, 960], ylabel='posX (pixel)')
    plot1D( cy, m='s', c='r',ax=ax[1], legend='mass center-refl_Y', 
           ylim=[1540, 1544], xlabel='frames',ylabel='posY (pixel)')

_record_xpcs_cell_timing("59 - Calculate ROI mass center", _xpcs_cell_start)


## One time Correlation

Note : Enter the number of buffers for Muliti tau one time correlation
number of buffers has to be even. More details in https://github.com/scikit-beam/scikit-beam/blob/master/skbeam/core/correlation.py

### if define another good_series

In [ ]:
# Select one-time correlation range

_xpcs_cell_start = time.time()

good_start_g2 = FD.beg
good_end_g2 = FD.end
define_good_series = False
hdf5_name_extension=''
if 'rstop' in locals() or 'rstart' in locals():
    define_good_series = True
if define_good_series:
    if 'rstart' in locals():
        good_start_g2 = rstart
    if 'rstop' in locals():
        good_end_g2 = rstop
    hdf5_name_extension = '_%s_%s'%(good_start_g2,good_end_g2)
    FD = Multifile(filename, beg = good_start_g2, end = good_end_g2) #end=1000)
    uid_ = uidstr + '_fra_%s_%s'%(FD.beg, FD.end)
    print( uid_ )

_record_xpcs_cell_timing("60 - Select one-time correlation range", _xpcs_cell_start)


In [ ]:
# Build correlation normalization

_xpcs_cell_start = time.time()

if use_sqnorm:#for transmision SAXS
    norm = get_pixelist_interp_iq( qp_saxs, iq_saxs, roi_mask, setup_pargs['center'])
    print('Using circular average in the normalization of G2 for SAXS scattering.')
elif use_SG:#for Gi-SAXS or WAXS
    avg_imgf = sgolay2d( avg_img, window_size= 11, order= 5) * mask
    norm=np.ravel(avg_imgf)[pixelist]    
    print('Using smoothed image by SavitzkyGolay filter in the normalization of G2.')      
elif use_SG_bin_frames:#for Gi-SAXS or WAXS
    avg_imgf = sgolay2d( avg_img, window_size= 11, order= 5) * mask
    norm_avg=np.ravel(avg_imgf)[pixelist]    
    #print('Using smoothed image by SavitzkyGolay filter in the normalization of G2.')
    bins_number = 1 
    norm = get_SG_norm( FD, pixelist, bins=bins_number, mask=mask, window_size= 11, order=5 )   
    print('Using smoothed bined: %s frames  by SavitzkyGolay filter in the normalization of G2.'%bins_number)     
else:     
    norm= None
    print('Using simple (average) normalization of G2.')      

if use_imgsum_norm:
    imgsum_ = imgsum
    print('Using frame total intensity for intensity normalization in g2 calculation.')      
else:
    imgsum_ = None

_record_xpcs_cell_timing("61 - Build correlation normalization", _xpcs_cell_start)


In [ ]:
# Calculate one-time correlation

_xpcs_cell_start = time.time()

if run_one_time: 
    if cal_g2_error:          
        g2,lag_steps,g2_err = cal_g2p(FD,roi_mask,bad_frame_list,good_start, num_buf = 8,
                            num_lev= None,imgsum= imgsum_, norm=norm, cal_error= True )
    else:   
        g2,lag_steps    =     cal_g2p(FD,roi_mask,bad_frame_list,good_start, num_buf = 8,
                            num_lev= None,imgsum= imgsum_, norm=norm, cal_error= False )

_record_xpcs_cell_timing("62 - Calculate one-time correlation", _xpcs_cell_start)


In [ ]:
# Save one-time correlation

_xpcs_cell_start = time.time()

if run_one_time:
    
    taus = lag_steps * timeperframe         
    try:
        g2_pds = save_g2_general( g2, taus=taus,qr= np.array( list( qval_dict.values() ) )[:g2.shape[1],0],
                                            qz = np.array( list( qval_dict.values() ) )[:g2.shape[1],1],
                             uid=uid_+'_g2.csv', path= data_dir, return_res=True )
    except:
        g2_pds = save_g2_general( g2, taus=taus,qr= np.array( list( qval_dict.values() ) )[:g2.shape[1],0],                                             
                             uid=uid_+'_'+q_mask_name+'_g2.csv', path= data_dir, return_res=True )   
    if cal_g2_error:    
        try:
            g2_err_pds = save_g2_general( g2_err, taus=taus,qr= np.array( list( qval_dict.values() ) )[:g2.shape[1],0],
                                                qz = np.array( list( qval_dict.values() ) )[:g2.shape[1],1],
                                 uid=uid_+'_g2_err.csv', path= data_dir, return_res=True )
        except:
            g2_err_pds = save_g2_general( g2_err, taus=taus,qr= np.array( list( qval_dict.values() ) )[:g2.shape[1],0],                                             
                                 uid=uid_+'_'+q_mask_name+'_g2_err.csv', path= data_dir, return_res=True )

_record_xpcs_cell_timing("63 - Save one-time correlation", _xpcs_cell_start)


# Fit g2

In [ ]:
# Fit one-time correlation

_xpcs_cell_start = time.time()

if run_one_time:
    try:
        g2_fit_result, taus_fit, g2_fit = get_g2_fit_general( g2,  taus, 
                    function = fit_g2_func,  vlim=[0.95, 1.05], fit_range= None,  
                fit_variables={'baseline':False, 'beta': True, 'alpha':True,'relaxation_rate':True,},                                  
                guess_values={'baseline':1.0,'beta': 0.2,'alpha':1.0,'relaxation_rate':1e-3},
                guess_limits = dict( baseline =[.9, 2], alpha=[0, 2],
                            beta = [0.01, 0.4], relaxation_rate= [1e-7, 1e7]) ,) 
        g2_fit_paras = save_g2_fit_para_tocsv(g2_fit_result,  filename= uid_ +'_'+q_mask_name +'_g2_fit_paras.csv', path=data_dir ) 
        g2_fit_failed = False
    except:
        g2_fit_failed = True
        from pandas import DataFrame # need to make up a fit result, otherwise writing the pdf report becomes difficult
        g2_fit_paras=DataFrame(np.zeros([np.shape(g2)[1],4]),columns=['beta','relaxation_rate','alpha','baseline'])

_record_xpcs_cell_timing("64 - Fit one-time correlation", _xpcs_cell_start)


In [ ]:
# Plot one-time correlation fit

_xpcs_cell_start = time.time()

if run_one_time:
    if cal_g2_error:
        g2_fit_err = np.zeros_like(g2_fit)
        if not g2_fit_failed:
            
            plot_xpcs_correlation( g2_dict={1:g2, 2:g2_fit}, taus_dict={1:taus, 2:taus_fit},
                    vlim=[0.95, 1.05], g2_err_dict= {1:g2_err, 2: g2_fit_err},
                    qval_dict = dict(itertools.islice(qval_dict.items(),g2.shape[1])), fit_res= g2_fit_result,  geometry=scat_geometry_, mode=scat_geometry,filename= uid_+'_g2', 
                    path= data_dir, function= fit_g2_func,  ylabel='g2', append_name=  '_fit',
                           qpi_analysis=True)
        else:
            plot_xpcs_correlation(g2_dict={1:g2, }, taus_dict={1:taus,},
                    vlim=[0.95, 1.05], g2_err_dict= {1:g2_err},
                    qval_dict = dict(itertools.islice(qval_dict.items(),g2.shape[1])),
                        fit_res= None,  geometry=scat_geometry_, mode=scat_geometry,filename= uid_+'_g2', 
                    path= data_dir, function= fit_g2_func,  ylabel='g2', append_name=  '',
                           qpi_analysis=True)
            
    else:    
        if not g2_fit_failed:
            plot_xpcs_correlation( g2_dict={1:g2, 2:g2_fit}, taus_dict={1:taus, 2:taus_fit}, vlim=[0.95, 1.05],
                qval_dict = dict(itertools.islice(qval_dict.items(),g2.shape[1])), fit_res= g2_fit_result,  geometry=scat_geometry_, mode=scat_geometry,filename= uid_+'_g2', 
                path= data_dir, function= fit_g2_func,  ylabel='g2', append_name=  '_fit',
                           qpi_analysis=True)
        else:    
            plot_xpcs_correlation( g2_dict={1:g2}, taus_dict={1:taus}, vlim=[0.95, 1.05],
                qval_dict = dict(itertools.islice(qval_dict.items(),g2.shape[1])), fit_res= None,  geometry=scat_geometry_, mode=scat_geometry,filename= uid_+'_g2', 
                path= data_dir, function= fit_g2_func,  ylabel='g2', append_name=  '',
                           qpi_analysis=True)

_record_xpcs_cell_timing("65 - Plot one-time correlation fit", _xpcs_cell_start)


In [ ]:
# Analyze one-time relaxation rates

_xpcs_cell_start = time.time()

if run_one_time and scat_geometry != 'waxs':
    try:
        if True:
            fs, fe = 0, 6
            #fs,fe=0, 6
            qval_dict_ = {k:qval_dict[k] for k in list(qval_dict.keys())[fs:fe]  }
            D0, qrate_fit_res = get_q_rate_fit_general(  qval_dict_, g2_fit_paras['relaxation_rate'][fs:fe], 
                                                       geometry=  scat_geometry_ )
            plot_q_rate_fit_general( qval_dict_, g2_fit_paras['relaxation_rate'][fs:fe],  qrate_fit_res, 
                                    geometry= scat_geometry_,uid=uid_  , path= data_dir )
        elif False:
            D0, qrate_fit_res = get_q_rate_fit_general(  qval_dict, g2_fit_paras['relaxation_rate'],
                                        fit_range=[0, 26],   geometry= scat_geometry_ )    
            plot_q_rate_fit_general( qval_dict, g2_fit_paras['relaxation_rate'],  qrate_fit_res,   
                                geometry=  scat_geometry_,uid=uid_  ,
                                    show_fit=False, path= data_dir, plot_all_range=False) 
        else:
            ##plot log_q, log_rate
            plot_q_rate_general( qval_dict, g2_fit_paras['relaxation_rate'],  
                                    geometry= scat_geometry_,uid=uid_  , path= data_dir )

            plot_q_g2fitpara_general( qval_dict, g2_fit_paras,  
                                    geometry= scat_geometry_,uid=uid_  , path= data_dir )
    except:
        pass

if run_one_time and scat_geometry == 'waxs':
    plot_roi_index_rates(g2_fit_paras['relaxation_rate'], data_dir, uid_)

_record_xpcs_cell_timing("66 - Analyze one-time relaxation rates", _xpcs_cell_start)


In [ ]:
# Update one-time progress

_xpcs_cell_start = time.time()

if track_progress:
    if settings_dict_['run_one_time']:
        update_prog_dict_run_XPCS_SAXS(prog_dict_file,process_id,progress_key='run_one_time', progress='stop',start_time=None, stop_time=time.time(), verbose=True)
    if settings_dict_['run_two_time']:
        update_prog_dict_run_XPCS_SAXS(prog_dict_file,process_id,progress_key='run_two_time', progress='start',start_time=time.time(), stop_time=None, verbose=True)

_record_xpcs_cell_timing("67 - Update one-time progress", _xpcs_cell_start)


# For two-time

In [ ]:
# Select two-time correlation range

_xpcs_cell_start = time.time()

define_good_series = True  #making sure that the 2TCF takes ALL data into account -> this should be the default!
if define_good_series:
    good_start = 0
    good_end =md['number of images']
    FD = Multifile(filename, beg = good_start, end = good_end)
    uid_ = uidstr + '_fra_%s_%s'%(FD.beg, FD.end)
    print( uid_ )

_record_xpcs_cell_timing("68 - Select two-time correlation range", _xpcs_cell_start)


In [ ]:
# Load two-time pixel data

_xpcs_cell_start = time.time()

data_pixel = None
if run_two_time:    
    data_pixel =   Get_Pixel_Arrayc( FD, pixelist,  norm= norm,  ).get_data()
    
    ### With another normalizaiton by using roi_intensity for each frame
    #data_pixel =   Get_Pixel_Arrayc( FD, pixelist,  norm= norm,  mean_int_sets = mean_int_setsF,qind=qind ).get_data()

_record_xpcs_cell_timing("69 - Load two-time pixel data", _xpcs_cell_start)


In [ ]:
# Calculate two-time correlation

_xpcs_cell_start = time.time()

g12b=None
if run_two_time:     
    g12b = auto_two_Arrayc(  data_pixel,  roi_mask, index = None   )
    if run_dose:
        np.save( data_dir + 'uid=%s_g12b'%uid, g12b)

_record_xpcs_cell_timing("70 - Calculate two-time correlation", _xpcs_cell_start)


In [ ]:
# Plot two-time correlation

_xpcs_cell_start = time.time()

if run_two_time:
    
    show_C12(  g12b , q_ind=min(6, g12b.shape[2]), qlabel=dict(itertools.islice(qval_dict.items(),g12b.shape[2])),
             #N1= FD.beg,logs=False, N2=min( FD.end,10000), 
             vmin= 1.00, vmax=1.25,  timeperframe=timeperframe,save=True, 
             path= data_dir, uid = uid_ ,cmap=plt.cm.jet)#cmap=cmap_albula)

_record_xpcs_cell_timing("71 - Plot two-time correlation", _xpcs_cell_start)


In [ ]:
# Derive one-time correlation from two-time data

_xpcs_cell_start = time.time()

multi_tau_steps = True
if run_two_time:
    if lag_steps is None:
        num_bufs=8
        noframes = FD.end - FD.beg
        num_levels = int(np.log( noframes/(num_bufs-1))/np.log(2) +1) +1
        tot_channels, lag_steps, dict_lag = multi_tau_lags(num_levels, num_bufs)
        max_taus= lag_steps.max()
        
    #max_taus= lag_steps.max()  
    max_taus = Nimg    
    #tausb = np.arange( g2b.shape[0])[:max_taus] *timeperframe
    if multi_tau_steps:
        lag_steps_ = lag_steps[   lag_steps <= g12b.shape[0] ]
        g2b = get_one_time_from_two_time(g12b)[lag_steps_]
        tausb = lag_steps_ *timeperframe
    else:
        tausb = (np.arange( g12b.shape[0]) *timeperframe)[:-200]
        g2b = (get_one_time_from_two_time(g12b))[:-200]
         
    g2b_pds = save_g2_general( g2b, taus=tausb, qr= np.array( list( qval_dict.values() ) )[:g2b.shape[1],0],
                              qz=(np.array(list(qval_dict.values()))[:g2b.shape[1],1] if scat_geometry in ('gi_saxs', 'gi_waxs') else None),
                              uid=uid_+'_'+q_mask_name+'_g2b.csv', path= data_dir, return_res=True )

_record_xpcs_cell_timing("72 - Derive one-time correlation from two-time data", _xpcs_cell_start)


In [ ]:
# Fit derived correlations

_xpcs_cell_start = time.time()

if run_two_time:    
    try: 
        g2b_fit_result, tausb_fit, g2b_fit = get_g2_fit_general( g2b,  tausb, 
                function = fit_g2_func,  vlim=[0.95, 1.05], fit_range= None,  
            fit_variables={'baseline':False, 'beta': True, 'alpha':True,'relaxation_rate':True},                                  
            guess_values={'baseline':1.0,'beta': 0.15,'alpha':1.0,'relaxation_rate':1e-3,},
            guess_limits = dict( baseline =[1, 1.8], alpha=[0, 2],
                        beta = [0, 1], relaxation_rate= [1e-8, 5000]) ) 
        g2b_fit_paras = save_g2_fit_para_tocsv(g2b_fit_result,  filename= uid_  +'_'+q_mask_name+'_g2b_fit_paras.csv', path=data_dir )
        g2b_fit_failed = False
    except:
        g2b_fit_failed = True
        from pandas import DataFrame
        g2b_fit_paras = DataFrame(np.zeros([g2b.shape[1], 4]),
                                  columns=['beta', 'relaxation_rate', 'alpha', 'baseline'])

_record_xpcs_cell_timing("73 - Fit derived correlations", _xpcs_cell_start)


In [ ]:
# Plot derived correlation fit

_xpcs_cell_start = time.time()

if run_two_time:
    if not g2b_fit_failed:
        plot_xpcs_correlation( g2_dict={1:g2b, 2:g2b_fit}, taus_dict={1:tausb, 2:tausb_fit}, vlim=[0.95, 1.05],
                qval_dict=dict(itertools.islice(qval_dict.items(),g2b.shape[1])), fit_res= g2b_fit_result,  geometry=scat_geometry_, mode=scat_geometry,filename=uid_+'_g2', 
                    path= data_dir, function= fit_g2_func,  ylabel='g2', append_name=  '_b_fit')
    else:
        plot_xpcs_correlation( g2_dict={1:g2b,}, taus_dict={1:tausb, }, vlim=[0.95, 1.05],
                         qval_dict=dict(itertools.islice(qval_dict.items(),g2b.shape[1])), fit_res= None,
                         geometry=scat_geometry_, mode=scat_geometry,filename=uid_+'_g2', 
                         path= data_dir, function= fit_g2_func,  ylabel='g2', append_name=  '')

_record_xpcs_cell_timing("74 - Plot derived correlation fit", _xpcs_cell_start)


In [ ]:
# Fit derived relaxation rates

_xpcs_cell_start = time.time()

if run_two_time and not g2b_fit_failed and scat_geometry != 'waxs' and len(qval_dict) >= 2:
    D0b, qrate_fit_resb = get_q_rate_fit_general(  dict(itertools.islice(qval_dict.items(),g2b.shape[1])), g2b_fit_paras['relaxation_rate'],
                                        fit_range=[0, 10],  geometry= scat_geometry_ )

_record_xpcs_cell_timing("75 - Fit derived relaxation rates", _xpcs_cell_start)


In [ ]:
# Analyze derived relaxation rates

_xpcs_cell_start = time.time()

if run_two_time and scat_geometry != 'waxs':
    try:
        if True:
            fs, fe = 0, 8
            #fs,fe=0, 6
            qval_dict_ = {k:qval_dict[k] for k in list(qval_dict.keys())[fs:fe]  }
            D0, qrate_fit_res = get_q_rate_fit_general(  qval_dict_, g2b_fit_paras['relaxation_rate'][fs:fe], 
                                                       geometry=  scat_geometry_ )
            plot_q_rate_fit_general( qval_dict_, g2b_fit_paras['relaxation_rate'][fs:fe],  qrate_fit_res, 
                                    geometry= scat_geometry_,uid=uid_ +'_two_time' , path= data_dir )
        elif False:
            D0, qrate_fit_res = get_q_rate_fit_general(  qval_dict, g2b_fit_paras['relaxation_rate'],
                                        fit_range=[0, 26],   geometry= scat_geometry_ )    
            plot_q_rate_fit_general( qval_dict, g2_fit_paras['relaxation_rate'],  qrate_fit_res,   
                                geometry=  scat_geometry_,uid=uid_ +'_two_time' ,
                                    show_fit=False, path= data_dir, plot_all_range=False) 
        else:
            ##plot log_q, log_rate
            plot_q_rate_general( qval_dict, g2b_fit_paras['relaxation_rate'],  
                                    geometry= scat_geometry_,uid=uid_ +'_two_time' , path= data_dir )

            plot_q_g2fitpara_general( qval_dict, g2b_fit_paras,  
                                    geometry= scat_geometry_,uid=uid_ +'_two_time' , path= data_dir )
    except:
        pass

if run_two_time and scat_geometry == 'waxs':
    plot_roi_index_rates(g2b_fit_paras['relaxation_rate'], data_dir, uid_+'_two_time')

_record_xpcs_cell_timing("76 - Analyze derived relaxation rates", _xpcs_cell_start)


In [ ]:
# Compare one- and two-time correlations

_xpcs_cell_start = time.time()

if run_two_time and run_one_time:
    plot_xpcs_correlation( g2_dict={1:g2, 2:g2b}, taus_dict={1:taus, 2:tausb},vlim=[0.99, 1.007],
                qval_dict=dict(itertools.islice(qval_dict.items(),g2.shape[1])), g2_labels=['from_one_time', 'from_two_time'],
            geometry=scat_geometry_, mode=scat_geometry,filename=uid_+'_g2_two_g2', path= data_dir, ylabel='g2', )

_record_xpcs_cell_timing("77 - Compare one- and two-time correlations", _xpcs_cell_start)


# Run Dose dependent analysis

In [ ]:
# Run dose-dependent two-time analysis

_xpcs_cell_start = time.time()

if run_dose:
    get_two_time_mulit_uids( [uid], roi_mask,  norm= norm,  bin_frame_number=1, 
                        path= data_dir0, force_generate=False, compress_path =  cmp_path + '/' )

_record_xpcs_cell_timing("78 - Run dose-dependent two-time analysis", _xpcs_cell_start)


In [ ]:
# Print transmission

_xpcs_cell_start = time.time()

try:
    print( 'transmission: %s  exposure time: %s [s]'%(md['transmission'],exposuretime) )
except:
    md['transmission'] =1
    print( 'transmission: %s  exposure time: %s [s]'%(md['transmission'],exposuretime) )

_record_xpcs_cell_timing("79 - Print transmission", _xpcs_cell_start)


In [ ]:
# Print dose

_xpcs_cell_start = time.time()

if run_dose:
    N = len(imgs)
    print(N)
    #exposure_dose = md['transmission'] * exposuretime* np.int_([     N/16, N/8, N/4 ,N/2, 3*N/4, N*0.99 ])
    exposure_dose = md['transmission'] * exposuretime* np.int_([      N/8, N/4 ,N/2, 3*N/4, N*0.99 ])
    
    print( exposure_dose )

_record_xpcs_cell_timing("80 - Print dose", _xpcs_cell_start)


In [ ]:
# Calculate dose-dependent correlations

_xpcs_cell_start = time.time()

if run_dose:
    taus_uids, g2_uids = get_series_one_time_mulit_uids( [ uid ],  qval_dict, good_start=good_start,  
                    path= data_dir0, exposure_dose = exposure_dose,  num_bufs =8, save_g2= False,
                                                   dead_time = 0, trans = [ md['transmission'] ] )

_record_xpcs_cell_timing("81 - Calculate dose-dependent correlations", _xpcs_cell_start)


In [ ]:
# Plot dose-dependent correlations

_xpcs_cell_start = time.time()

if run_dose:    
    plot_dose_g2( taus_uids, g2_uids, ylim=[1.0, 1.2], vshift= 0.00,
                 qval_dict = qval_dict, fit_res= None,  geometry= scat_geometry_,
                 filename= '%s_dose_analysis'%uid_, 
                path= data_dir, function= None,  ylabel='g2_Dose', g2_labels= None, append_name=  '' )

_record_xpcs_cell_timing("82 - Plot dose-dependent correlations", _xpcs_cell_start)


In [ ]:
# Plot selected dose-dependent correlation

_xpcs_cell_start = time.time()

if run_dose:
    qth_interest = min(5, len(qval_dict))
    plot_dose_g2( taus_uids, g2_uids, qth_interest= qth_interest, ylim=[0.98, 1.4], vshift= 0.00,
                 qval_dict = qval_dict, fit_res= None,  geometry= scat_geometry_,
                 filename= '%s_dose_analysis'%uidstr, 
                path= data_dir, function= None,  ylabel='g2_Dose', g2_labels= None, append_name=  '' )

_record_xpcs_cell_timing("83 - Plot selected dose-dependent correlation", _xpcs_cell_start)


In [ ]:
# Update two-time progress

_xpcs_cell_start = time.time()

if track_progress:
    if settings_dict_['run_two_time']:
        update_prog_dict_run_XPCS_SAXS(prog_dict_file,process_id,progress_key='run_two_time', progress='stop',start_time=None, stop_time=time.time(), verbose=True)
    if settings_dict_['xsvs']:
        update_prog_dict_run_XPCS_SAXS(prog_dict_file,process_id,progress_key='xsvs', progress='start',start_time=time.time(), stop_time=None, verbose=True)

_record_xpcs_cell_timing("84 - Update two-time progress", _xpcs_cell_start)


# Four Time Correlation

In [ ]:
# Calculate four-time correlation

_xpcs_cell_start = time.time()

if run_four_time:
    g4 = get_four_time_from_two_time(g12b, g2=g2b)[:int(max_taus)]

_record_xpcs_cell_timing("85 - Calculate four-time correlation", _xpcs_cell_start)


In [ ]:
# Save four-time correlation

_xpcs_cell_start = time.time()

if run_four_time:
    taus4 = np.arange( g4.shape[0])*timeperframe        
    g4_pds = save_g2_general( g4, taus=taus4, qr=np.array( list( qval_dict.values() ) )[:,0],
                             qz=None, uid=uid_ +'_g4.csv', path= data_dir, return_res=True )

_record_xpcs_cell_timing("86 - Save four-time correlation", _xpcs_cell_start)


In [ ]:
# Plot four-time correlation

_xpcs_cell_start = time.time()

if run_four_time:
    plot_xpcs_correlation( g2_dict={1:g4}, taus_dict={1:taus4},vlim=[0.95, 1.05], qval_dict=qval_dict, fit_res= None, 
                geometry=scat_geometry_, mode=scat_geometry,filename=uid_+'_g4',path= data_dir,   ylabel='g4')

_record_xpcs_cell_timing("87 - Plot four-time correlation", _xpcs_cell_start)


# Speckle Visiblity

In [ ]:
# Prepare XSVS analysis

_xpcs_cell_start = time.time()

if run_xsvs:    
    max_cts = get_max_countc(FD, roi_mask )    
    #max_cts = 15 #for eiger 500 K
    qind, pixelist = roi.extract_label_indices(   roi_mask  )
    noqs = len( np.unique(qind) )
    nopr = np.bincount(qind, minlength=(noqs+1))[1:]
    #time_steps = np.array( utils.geometric_series(2,   len(imgs)   ) )
    time_steps = [0,1]  #only run the first two levels
    num_times = len(time_steps)    
    times_xsvs = exposuretime + (2**(  np.arange( len(time_steps) ) ) -1 ) * timeperframe   
    print( 'The max counts are: %s'%max_cts )

_record_xpcs_cell_timing("88 - Prepare XSVS analysis", _xpcs_cell_start)


### Do histogram 

In [ ]:
# Calculate XSVS histograms

_xpcs_cell_start = time.time()

if run_xsvs:
    if roi_avg is  None:
        times_roi, mean_int_sets = cal_each_ring_mean_intensityc(FD, roi_mask, timeperframe = None,  ) 
        roi_avg = np.average( mean_int_sets, axis=0)
    
    spec_bins, spec_his, spec_std, spec_sum  =  xsvsp( FD, np.int_(roi_mask), norm=None,
                max_cts=int(max_cts+2),  bad_images=bad_frame_list, only_two_levels=True )    
    spec_kmean =  np.array(  [roi_avg * 2**j for j in  range( spec_his.shape[0] )] )
    spec_pds =  save_bin_his_std( spec_bins, spec_his, spec_std, filename=uid_+'_spec_res.csv', path=data_dir )

_record_xpcs_cell_timing("89 - Calculate XSVS histograms", _xpcs_cell_start)


### Do historam fit by negtive binominal function with maximum likehood method

In [ ]:
# Fit XSVS histograms

_xpcs_cell_start = time.time()

if run_xsvs:    
    ML_val, KL_val,K_ = get_xsvs_fit(  spec_his, spec_sum, spec_kmean, 
                        spec_std, max_bins=2, fit_range=[1,60], varyK= False  )
    #print( 'The observed average photon counts are: %s'%np.round(K_mean,4))
    #print( 'The fitted average photon counts are: %s'%np.round(K_,4)) 
    print( 'The difference sum of average photon counts between fit and data are: %s'%np.round( 
            abs(np.sum( spec_kmean[0,:] - K_ )),4))
    print( '#'*30)
    qth=   0 
    print( 'The fitted M for Qth= %s are: %s'%(qth, ML_val[qth]) )
    print( K_[qth])
    print( '#'*30)

_record_xpcs_cell_timing("90 - Fit XSVS histograms", _xpcs_cell_start)


## Plot fit results

In [ ]:
# Plot XSVS fits

_xpcs_cell_start = time.time()

if run_xsvs:   
    qr = [qval_dict[k][0] for k in list(qval_dict.keys()) ]
    plot_xsvs_fit(  spec_his, ML_val, KL_val, K_mean = spec_kmean, spec_std=spec_std,
                  xlim = [0,10], vlim =[.9, 1.1],
        uid=uid_, qth= qth_interest, logy= True, times= times_xsvs, q_ring_center=qr, path=data_dir)
    
    plot_xsvs_fit(  spec_his, ML_val, KL_val, K_mean = spec_kmean, spec_std = spec_std,
                  xlim = [0,15], vlim =[.9, 1.1],
        uid=uid_, qth= None, logy= True, times= times_xsvs, q_ring_center=qr, path=data_dir )

_record_xpcs_cell_timing("91 - Plot XSVS fits", _xpcs_cell_start)


### Get contrast

In [ ]:
# Calculate XSVS contrast

_xpcs_cell_start = time.time()

if run_xsvs:
    contrast_factorL = get_contrast( ML_val)
    spec_km_pds = save_KM(  spec_kmean, KL_val, ML_val, qs=qr, level_time=times_xsvs, uid=uid_, path = data_dir )
    #spec_km_pds

_record_xpcs_cell_timing("92 - Calculate XSVS contrast", _xpcs_cell_start)


### Plot contrast with g2 results

In [ ]:
# Compare XSVS contrast and correlation

_xpcs_cell_start = time.time()

if run_xsvs:    
    plot_g2_contrast( contrast_factorL, g2b, times_xsvs, tausb, qr, 
                     vlim=[0.8,1.2], qth = qth_interest, uid=uid_,path = data_dir, legend_size=14)

    plot_g2_contrast( contrast_factorL, g2b, times_xsvs, tausb, qr, 
                     vlim=[0.8,1.2], qth = None, uid=uid_,path = data_dir, legend_size=4)

_record_xpcs_cell_timing("93 - Compare XSVS contrast and correlation", _xpcs_cell_start)


In [ ]:
# Update XSVS progress

_xpcs_cell_start = time.time()

if track_progress:
    if settings_dict_['xsvs']:
        update_prog_dict_run_XPCS_SAXS(prog_dict_file,process_id,progress_key='xsvs', progress='stop',start_time=None, stop_time=time.time(), verbose=True)
    update_prog_dict_run_XPCS_SAXS(prog_dict_file,process_id,progress_key='save data and report', progress='start',start_time=time.time(), stop_time=None, verbose=True)

_record_xpcs_cell_timing("94 - Update XSVS progress", _xpcs_cell_start)


# Export Results to a HDF5 File

In [ ]:
# Assemble and save result metadata

_xpcs_cell_start = time.time()
md['hostname'] = hostname[0]
md['kernel'] = kernel
md['detector_mask_file']=detector_mask_path+detector_mask_filename
md['det_mask_dict'] = det_mask_dict
md['polygon_mask']= polygon_dict
md['roi_mask_file']= fp
md['roi_source'] = roi_input_source
md['mask_source'] = mask_input_source
md['beam_mask_source'] = beam_mask_source
md['mask_file'] = mask_input_source
md['dynamic_mask_applied'] = dynamic_mask_applied
md['scat_geometry'] = scat_geometry
if scat_geometry in ('gi_saxs', 'gi_waxs'):
    md['surface_geometry'] = {
        'incident_angle_deg': surface_maps['incident_angle_deg'],
        'inplane_angle_deg': surface_maps['inplane_angle_deg'],
        'incident_center': surface_maps['incident_center'],
        'reflected_center': surface_maps['reflected_center'],
    }
    if scat_geometry == 'gi_saxs':
        md['surface_geometry']['profile_source'] = 'roi_mask' if use_roi_profiles else 'regular_q_grid'
    if surface_rois is not None:
        md['surface_geometry']['q_parallel_edges'] = surface_rois['parallel_edges']
        md['surface_geometry']['q_z_edges'] = surface_rois['z_edges']
    md['surface_reflected_center_x'] = refl_x0
    md['surface_reflected_center_y'] = refl_y0
md['mask'] = mask
#md['NOTEBOOK_FULL_PATH'] =  data_dir + get_current_pipeline_fullpath(NFP).split('/')[-1]
md['good_start'] = good_start
md['bad_frame_list'] = bad_frame_list
md['avg_img'] = avg_img
md['roi_mask'] = roi_mask
md['mask_outliers'] = mask_outliers
if mask_outliers:
    md['outlier_threshold'] = outlier_threshold
    md['maximum_outlier_fraction'] = maximum_outlier_fraction
md['setup_pargs'] = setup_pargs
if scat_geometry == 'gi_saxs':        
    if not use_roi_profiles:
        md['Qr'] = Qr
        md['Qz'] = Qz
    md['qval_dict'] = qval_dict
    md['beam_center_x'] =  inc_x0
    md['beam_center_y']=   inc_y0
    md['beam_refl_center_x'] = refl_x0
    md['beam_refl_center_y'] = refl_y0


elif scat_geometry == 'gi_waxs':
    md['qval_dict'] = qval_dict
    md['beam_center_x'] = inc_x0
    md['beam_center_y'] = inc_y0
    md['beam_refl_center_x'] = refl_x0
    md['beam_refl_center_y'] = refl_y0
elif scat_geometry == 'waxs':
    md['qval_dict'] = qval_dict
    md['beam_center_x'] = inc_x0
    md['beam_center_y'] = inc_y0
else:
    md['qr']= qr
    #md['qr_edge'] = qr_edge
    md['qval_dict'] = qval_dict
    md['beam_center_x'] =  center[1]
    md['beam_center_y']=  center[0]            

md['beg'] = FD.beg
md['end'] = FD.end
md['qth_interest'] = qth_interest
md['metadata_file'] = data_dir + 'uid=%s_md.pkl'%uid

md['beg_OneTime'] = good_start_g2
md['end_OneTime'] = good_end_g2

psave_obj(  md, data_dir + 'uid=%s_md.pkl'%uid ) #save the setup parameters
save_dict_csv( md,  data_dir + 'uid=%s_md.csv'%uid, 'w')

Exdt = {} 
if scat_geometry == 'gi_saxs':  
    for k,v in zip( ['md', 'roi_mask','qval_dict','avg_img','mask','pixel_mask', 'imgsum', 'bad_frame_list', 'qr_1d_pds'], 
                [md,    roi_mask, qval_dict, avg_img,mask,pixel_mask, imgsum, bad_frame_list, qr_1d_pds] ):
        Exdt[ k ] = v
elif scat_geometry == 'saxs': 
    for k,v in zip( ['md', 'q_saxs', 'iq_saxs','iqst','qt','roi_mask','qval_dict','avg_img','mask','pixel_mask', 'imgsum', 'bad_frame_list'], 
                [md, q_saxs, iq_saxs, iqst, qt,roi_mask, qval_dict, avg_img,mask,pixel_mask, imgsum, bad_frame_list] ):
        Exdt[ k ] = v
elif scat_geometry in ('gi_waxs', 'waxs'): 
    for k,v in zip( ['md', 'roi_mask','qval_dict','avg_img','mask','pixel_mask', 'imgsum', 'bad_frame_list'], 
                [md,       roi_mask, qval_dict, avg_img,mask,pixel_mask, imgsum, bad_frame_list] ):
        Exdt[ k ] = v
        
if scat_geometry in ('gi_saxs', 'gi_waxs') and surface_rois is not None:
    Exdt['q_parallel_edges'] = surface_rois['parallel_edges']
    Exdt['q_z_edges'] = surface_rois['z_edges']
if run_waterfall:Exdt['wat'] =  wat
if run_t_ROI_Inten:Exdt['times_roi'] = times_roi;Exdt['mean_int_sets']=mean_int_sets
if run_one_time:
    if run_invariant_analysis:
        for k,v in zip( ['taus','g2','g2_fit_paras', 'time_stamp','invariant'], [taus,g2,g2_fit_paras,time_stamp,invariant] ):Exdt[ k ] = v
    else:
        for k,v in zip( ['taus','g2','g2_fit_paras'  ], [taus,g2,g2_fit_paras ] ):Exdt[ k ] = v
            
if run_two_time:
    for k,v in zip( ['tausb','g2b','g2b_fit_paras', 'g12b'], [tausb,g2b,g2b_fit_paras,g12b] ):Exdt[ k ] = v
    #for k,v in zip( ['tausb','g2b','g2b_fit_paras', ], [tausb,g2b,g2b_fit_paras] ):Exdt[ k ] = v    
if run_dose:
    for k,v in zip( [ 'taus_uids', 'g2_uids' ], [taus_uids, g2_uids] ):Exdt[ k ] = v
if run_four_time:
    for k,v in zip( ['taus4','g4'], [taus4,g4] ):Exdt[ k ] = v
if run_xsvs:
    for k,v in zip( ['spec_kmean','spec_pds','times_xsvs','spec_km_pds','contrast_factorL'], 
                   [ spec_kmean,spec_pds,times_xsvs,spec_km_pds,contrast_factorL] ):Exdt[ k ] = v

_record_xpcs_cell_timing("95 - Assemble and save result metadata", _xpcs_cell_start)


In [ ]:
# Export HDF5 results

_xpcs_cell_start = time.time()

#export_xpcs_results_to_h5( 'uid=%s_%s_Res%s.h5'%(md['uid'],q_mask_name,hdf5_name_extension), data_dir, export_dict = Exdt )
export_xpcs_results_to_h5( 'uid=%s_%s%s_Res.h5'%(md['uid'],q_mask_name,hdf5_name_extension), data_dir, export_dict = Exdt )

_record_xpcs_cell_timing("96 - Export HDF5 results", _xpcs_cell_start)


In [ ]:
# Append result path to user list

_xpcs_cell_start = time.time()

try:
    from pyCHX.chx_generic_functions import append_txtfile
    outp = default_dir+'%s/%s/'%(md['cycle'],username)
    append_txtfile( filename =  outp  + username + '.lst', 
                   data =  np.array( [ outp +    'Results/' + md['uid'] + '/uid=%s__Res.h5'%md['uid']]))
except:
    print('ERROR: unable to append output textfile for this uid.')

_record_xpcs_cell_timing("97 - Append result path to user list", _xpcs_cell_start)


# Create a PDF Report

In [ ]:
# Determine PDF report name

_xpcs_cell_start = time.time()

pdf_out_dir = data_dir0

pdf_filename = "XPCS_Analysis_Report2_for_uid=%s%s%s.pdf"%(uid,pdf_version,q_mask_name)
if run_xsvs:
    pdf_filename = "XPCS_XSVS_Analysis_Report_for_uid=%s%s%s.pdf"%(uid,pdf_version,q_mask_name)

_record_xpcs_cell_timing("98 - Determine PDF report name", _xpcs_cell_start)


In [ ]:
# Create PDF report

_xpcs_cell_start = time.time()

# cheating here a little: the function argument 'False' hardcodes the absence of the form factor fit
make_pdf_report( data_dir, uid, pdf_out_dir, pdf_filename, username, 
                   False,run_one_time, run_two_time, run_four_time, run_xsvs, run_dose,
                report_type= ('gi_waxs' if scat_geometry == 'waxs' else scat_geometry), report_invariant= run_invariant_analysis,
               md = md )

_record_xpcs_cell_timing("99 - Create PDF report", _xpcs_cell_start)


In [ ]:
# Finish report progress tracking

_xpcs_cell_start = time.time()

if track_progress:
    update_prog_dict_run_XPCS_SAXS(prog_dict_file,process_id,progress_key='save data and report', progress='stop',start_time=None, stop_time=time.time(), verbose=True)

_record_xpcs_cell_timing("100 - Finish report progress tracking", _xpcs_cell_start)


## Attach the PDF report to Olog 

In [ ]:
# Attach PDF report to Olog

_xpcs_cell_start = time.time()

if att_pdf_report and enable_olog_attachments:
    lops=0
    while lops<5:
        try:
            update_olog_uid_with_file( uid[:6], text='Add XPCS Analysis PDF Report', 
                              filename=pdf_out_dir + pdf_filename, append_name='_R1' )
            lops=5
        except:
            print("Failed to attach PDF report to Olog...try again in 10s")
            time.sleep(10)
            lops+=1
            if lops == 5:
                print("Failed final attempt to attach PDF to Olog!")
if not enable_olog_attachments:
    print('Olog attachments are NOT possible as this notebook (or the papermill that is calling it) is NOT running on saturn4.\nConnection to Olog is blocked by firewall, so we are not going to waste time trying.')

_record_xpcs_cell_timing("101 - Attach PDF report to Olog", _xpcs_cell_start)


## Save the OAV images

In [ ]:
# Attach OAV images to Olog

_xpcs_cell_start = time.time()

if save_oavs:
    try:
        #os.environ['HTTPS_PROXY'] = 'https://proxy:8888'
        #os.environ['no_proxy'] = 'cs.nsls2.local,localhost,127.0.0.1'
        update_olog_uid_with_file( uid[:6], text='Add OVA images', 
                              filename= data_dir + 'uid=%s_OVA_images.png'%uid, append_name='_img' )
    except:
        print('failed to update Olog with OAV images')

_record_xpcs_cell_timing("102 - Attach OAV images to Olog", _xpcs_cell_start)


# The End!

In [ ]:
# Prepare timing report

_xpcs_cell_start = time.time()
_xpcs_timing_report_heading = "XPCS NOTEBOOK CELL TIMING REPORT"
_record_xpcs_cell_timing("103 - Prepare timing report", _xpcs_cell_start)


In [ ]:
# Complete and print timing report

_xpcs_cell_start = time.time()
_record_xpcs_cell_timing("104 - Complete and print timing report", _xpcs_cell_start)
_xpcs_notebook_elapsed = time.time() - _xpcs_notebook_start
_xpcs_timed_total = sum(xpcs_cell_timings.values())

print("\n" + "=" * 80)
print(_xpcs_timing_report_heading)
print("=" * 80)
for _xpcs_label, _xpcs_elapsed in xpcs_cell_timings.items():
    print(f"{_xpcs_label:<65} {_xpcs_elapsed:>10.2f} s")
print("-" * 80)
print(f"{'Timed cells total':<65} {_xpcs_timed_total:>10.2f} s")
print(f"{'Notebook wall time':<65} {_xpcs_notebook_elapsed:>10.2f} s")
print("=" * 80)
